# Applied Data Science Capstone — 2. Data Wrangling & Exploratory Analysis

## Objective
Prepare the collected launch records for analysis:

1. Inspect the frame (dtypes, missing values, value ranges).
2. Count launches per launch site and landing-outcome frequencies.
3. Derive the binary target **`Class`** (1 = first stage landed successfully, 0 = otherwise) and verify the **67% landing success rate**.
4. Persist the cleaned frame for the SQL / visualisation / ML notebooks.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/dataset_spacex.csv")
print("shape:", df.shape)
print()
print(df.dtypes.to_string())
print()
print("missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0].to_string() or "none")

shape: (90, 23)

FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
LandingType        object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
RecoveryShip       object
Block               int64
ReusedCount         int64
Serial             object
Customer           object
StaticFire           bool
MissionOutcome     object
Latitude          float64
Longitude         float64
Class               int64
Outcome            object

missing values per column:
LandingType     11
LandingPad      65
RecoveryShip    53


### Launch counts per site (Falcon 9 only)

In [2]:
site_counts = df["LaunchSite"].value_counts()
print(site_counts.to_string())
print()
print("total:", site_counts.sum())
# Reference: the raw Spacex.csv used in the course holds 101 rows with
# CCAFS LC-40: 26, CCAFS SLC-40: 34, KSC LC-39A: 25, VAFB SLC-4E: 16.
# Filtering to Falcon 9 only leaves the 90 launches analysed here.

LaunchSite
CCAFS SLC-40    30
CCAFS LC-40     23
KSC LC-39A      22
VAFB SLC-4E     15

total: 90


### Landing outcomes

In [3]:
outcome_counts = df["Outcome"].value_counts()
print(outcome_counts.to_string())
print()
success = (df["Class"] == 1).sum()
total = len(df)
print(f"successful first-stage landings: {success}/{total} = {success/total:.1%}")

Outcome
True ASDS      33
True RTLS      23
False Ocean    13
None None      11
True Ocean      4
False ASDS      4
False RTLS      2

successful first-stage landings: 60/90 = 66.7%


### Target variable `Class`
`Class = 1` when the booster landed successfully (`True RTLS` / `True ASDS` /
`True Ocean` outcomes), else `0`. The 67% success rate matches the figure
reported in the course materials.

In [4]:
print(df.groupby("Outcome")["Class"].agg(["count", "mean"]).to_string())
print()
print("Payload mass summary (kg):")
print(df["PayloadMass"].describe().to_string())
print()
print("Orbits present:", sorted(df["Orbit"].unique()))

             count  mean
Outcome                 
False ASDS       4   0.0
False Ocean     13   0.0
False RTLS       2   0.0
None None       11   0.0
True ASDS       33   1.0
True Ocean       4   1.0
True RTLS       23   1.0

Payload mass summary (kg):
count       90.000000
mean      4560.837778
std       3652.387121
min        639.200000
25%       2169.500000
50%       3391.950000
75%       5756.000000
max      15600.000000

Orbits present: ['ES-L1', 'GEO', 'GTO', 'HEO', 'ISS', 'LEO', 'MEO', 'PO', 'SSO', 'VLEO']


### Persist the cleaned dataset

In [5]:
df.to_csv("../data/dataset_spacex.csv", index=False)  # idempotent clean save
print("saved ../data/dataset_spacex.csv with", df.shape[0], "rows")

saved ../data/dataset_spacex.csv with 90 rows
